This notebook is having code for the manuscript *Predictive Modelling of Conducting Casson Nanofluid Flow Using a Physics-Informed Neural Network: Effects of Thermal Stratification and Coupled Transport Mechanisms*

In [ ]:
# 1. Imports and reproducibility
import os, random, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp

import torch
import torch.nn as nn

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
print("Device:", device)
print("PyTorch:", torch.__version__)


## 2. Parameter settings

In [2]:
BASE = {
    "beta": 0.5,  # Casson parameter
    "M": 0.3,     # magnetic parameter; Table 2 baseline group
    "Kp": 0.4,    # porous-medium parameter; Table 2 baseline group
    "Pr": 1.1,    # Prandtl number
    "Rd": 0.3,    # thermal radiation
    "Ec": 0.1,    # Eckert number
    "Nb": 0.1,    # Brownian motion
    "Nt": 0.1,    # thermophoresis
    "St": 0.1,    # thermal stratification
    "Q": 0.1,     # heat-generation parameter
    "Sc": 0.2,    # Schmidt number
    "c": 0.1,     # solutal stratification parameter appearing in Eq. (9)
    "Cr": 0.2,    # chemical reaction parameter
    "S": 0.2,     # suction parameter
}
ETA_MAX = 6.0      # manuscript states eta_max = 6
N_COLLOCATION = 200  # manuscript baseline; sensitivity experiment passes n_points explicitly
N_BVP_INITIAL = 300
BVP_TOL = 1e-8
BVP_MAX_NODES = 50000

# Eq. (10), as printed, specifies phi(0)=1-Sc.
# Set False ONLY if the manuscript is corrected to phi(0)=1-c.
WALL_PHI_USES_SC_AS_PRINTED = True

CASES = {
    "Baseline": {},
    "Thermal variation (Rd=1.5)": {"Rd": 1.5},
    "Concentration variation (Sc=0.6)": {"Sc": 0.6},
}


## 3. Governing equations implementation

In [3]:
def wall_phi_value(p):
    strat = p["Sc"] if WALL_PHI_USES_SC_AS_PRINTED else p["c"]
    return 1.0 - strat

def bvp_ode(eta, Y, p):
    # State vector: [f, f', f'', theta, theta', phi, phi']
    f, fp, fpp, th, thp, ph, php = Y

    # Eq. (7)
    fppp = (2.0*fp**2 + (p["M"] + p["Kp"])*fp - f*fpp) / (1.0 + 1.0/p["beta"])

    # Eq. (8)
    thpp = -(
        p["Pr"] * (
            f*thp - fp*th + p["St"]*fp + p["Q"]*th
            + p["Nb"]*thp*php + p["Nt"]*thp**2
        )
        + p["Pr"]*p["Ec"] * (
            (1.0 + 1.0/p["beta"])*fpp**2
            + (p["M"] + p["Kp"])*fp**2
        )
    ) / (1.0 + 4.0*p["Rd"]/3.0)

    # Eq. (9), transcribed literally: Sc multiplies f*phi' only.
    phpp = (
        -p["Sc"]*f*php + fp*ph + p["c"]*fp
        + p["Cr"]*ph**2 - (p["Nt"]/p["Nb"])*thpp
    )
    return np.vstack((fp, fpp, fppp, thp, thpp, php, phpp))


def bvp_bc(Ya, Yb, p):
    return np.array([
        Ya[0] - p["S"],
        Ya[1] - 1.0,
        Ya[3] - (1.0 - p["St"]),
        Ya[5] - wall_phi_value(p),
        Yb[1],
        Yb[3],
        Yb[5],
    ])


def initial_bvp_guess(eta, p):
    Y = np.zeros((7, eta.size))
    Y[0] = p["S"] + (1.0 - np.exp(-eta))
    Y[1] = np.exp(-eta)
    Y[2] = -np.exp(-eta)
    Y[3] = (1.0-p["St"])*np.exp(-eta)
    Y[4] = -(1.0-p["St"])*np.exp(-eta)
    Y[5] = wall_phi_value(p)*np.exp(-eta)
    Y[6] = -wall_phi_value(p)*np.exp(-eta)
    return Y


def solve_reference(p, eta_max=ETA_MAX, tol=BVP_TOL, n_initial=N_BVP_INITIAL):
    x = np.linspace(0.0, eta_max, n_initial)
    y0 = initial_bvp_guess(x, p)
    sol = solve_bvp(
        lambda x, y: bvp_ode(x, y, p),
        lambda ya, yb: bvp_bc(ya, yb, p),
        x, y0, tol=tol, max_nodes=BVP_MAX_NODES, verbose=0
    )
    if not sol.success:
        raise RuntimeError(f"solve_bvp failed for {p}: {sol.message}")
    return sol


def reference_wall_quantities(sol):
    return {
        "f''(0)": float(sol.y[2, 0]),
        "-theta'(0)": float(-sol.y[4, 0]),
        "-phi'(0)": float(-sol.y[6, 0]),
    }


## 4. Independent reference solver: convergence check

In [ ]:
reference_solutions = {}
reference_rows = []

for case_name, changes in CASES.items():
    p = BASE.copy()
    p.update(changes)
    sol = solve_reference(p)
    reference_solutions[case_name] = (p, sol)
    q = reference_wall_quantities(sol)
    reference_rows.append({
        "Case": case_name,
        **q,
        "BVP success": sol.success,
        "BVP nodes": int(sol.x.size),
        "BVP max RMS residual": float(np.max(sol.rms_residuals)),
        "Boundary residual max": float(np.max(np.abs(bvp_bc(sol.y[:, 0], sol.y[:, -1], p)))),
    })

reference_check = pd.DataFrame(reference_rows)
display(reference_check.round(8))


## 5. PINN model and residual loss

In [5]:
class PINN(nn.Module):
    def __init__(self, hidden_layers=6, neurons=64):
        super().__init__()
        layers = [nn.Linear(1, neurons), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(neurons, neurons), nn.Tanh()])
        layers.append(nn.Linear(neurons, 3))
        self.net = nn.Sequential(*layers)

    def forward(self, eta):
        # Input scaling improves conditioning; derivatives are still w.r.t. eta.
        z = 2.0*eta/ETA_MAX - 1.0
        return self.net(z)


def d_dx(y, x, create_graph=True):
    return torch.autograd.grad(
        y, x, grad_outputs=torch.ones_like(y),
        create_graph=create_graph, retain_graph=True
    )[0]


def pinn_residuals(model, p, eta):
    eta = eta.clone().detach().requires_grad_(True)
    out = model(eta)
    f, th, ph = out[:, 0:1], out[:, 1:2], out[:, 2:3]

    fp = d_dx(f, eta)
    fpp = d_dx(fp, eta)
    fppp = d_dx(fpp, eta)
    thp = d_dx(th, eta)
    thpp = d_dx(thp, eta)
    php = d_dx(ph, eta)
    phpp = d_dx(php, eta)

    r_f = (
        (1.0 + 1.0/p["beta"])*fppp + f*fpp
        - 2.0*fp**2 - p["M"]*fp - p["Kp"]*fp
    )
    r_th = (
        (1.0 + 4.0*p["Rd"]/3.0)*thpp
        + p["Pr"]*(f*thp - fp*th + p["St"]*fp + p["Q"]*th
                   + p["Nb"]*thp*php + p["Nt"]*thp**2)
        + p["Pr"]*p["Ec"]*((1.0 + 1.0/p["beta"])*fpp**2
                            + (p["M"] + p["Kp"])*fp**2)
    )
    r_ph = (
        phpp + p["Sc"]*f*php - fp*ph - p["c"]*fp
        - p["Cr"]*ph**2 + (p["Nt"]/p["Nb"])*thpp
    )
    return (r_f, r_th, r_ph)


def pinn_loss(model, p, n_points=None):
    if n_points is None:
        n_points = N_COLLOCATION
    eta = torch.linspace(0.0, ETA_MAX, n_points, device=device, dtype=DTYPE).reshape(-1, 1)
    residuals = pinn_residuals(model, p, eta)
    loss_ode = sum(torch.mean(r**2) for r in residuals)

    e0 = torch.zeros((1, 1), device=device, dtype=DTYPE, requires_grad=True)
    eL = torch.full((1, 1), ETA_MAX, device=device, dtype=DTYPE, requires_grad=True)
    o0, oL = model(e0), model(eL)
    fp0 = d_dx(o0[:, 0:1], e0)
    fpL = d_dx(oL[:, 0:1], eL)

    bc_terms = [
        o0[:, 0:1] - p["S"],
        fp0 - 1.0,
        o0[:, 1:2] - (1.0-p["St"]),
        o0[:, 2:3] - wall_phi_value(p),
        fpL,
        oL[:, 1:2],
        oL[:, 2:3],
    ]
    loss_bc = sum(torch.mean(b**2) for b in bc_terms)
    return loss_ode + 10.0*loss_bc, loss_ode.detach(), loss_bc.detach()


def train_pinn(p, adam_epochs=3000, lbfgs_iterations=200, verbose=True, n_points=None):
    model = PINN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    history = []
    start = time.time()

    for epoch in range(adam_epochs):
        opt.zero_grad()
        loss, loss_ode, loss_bc = pinn_loss(model, p, n_points=n_points)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        opt.step()
        if (epoch+1) % 250 == 0 or epoch == 0:
            history.append([epoch+1, float(loss.detach().cpu()),
                            float(loss_ode.cpu()), float(loss_bc.cpu())])
            if verbose:
                print(f"Adam {epoch+1}/{adam_epochs}: total={loss.item():.3e}, "
                      f"ODE={loss_ode.item():.3e}, BC={loss_bc.item():.3e}")

    opt2 = torch.optim.LBFGS(
        model.parameters(), lr=1.0, max_iter=lbfgs_iterations,
        history_size=50, line_search_fn="strong_wolfe",
        tolerance_grad=1e-9, tolerance_change=1e-12
    )
    def closure():
        opt2.zero_grad()
        loss, _, _ = pinn_loss(model, p, n_points=n_points)
        loss.backward()
        return loss
    opt2.step(closure)
    if verbose:
        final_loss, final_ode, final_bc = pinn_loss(model, p, n_points=n_points)
        print(f"L-BFGS final: total={final_loss.item():.3e}, "
              f"ODE={final_ode.item():.3e}, BC={final_bc.item():.3e}; "
              f"time={time.time()-start:.1f}s")
    return model, pd.DataFrame(history, columns=["epoch", "total_loss", "ODE_loss", "BC_loss"])


def pinn_wall_quantities(model):
    e0 = torch.zeros((1, 1), device=device, dtype=DTYPE, requires_grad=True)
    out = model(e0)
    f = out[:, 0:1]
    th = out[:, 1:2]
    ph = out[:, 2:3]
    fp = d_dx(f, e0)
    fpp = d_dx(fp, e0)
    thp = d_dx(th, e0)
    php = d_dx(ph, e0)
    return {
        "f''(0)": float(fpp.detach().cpu().item()),
        "-theta'(0)": float(-thp.detach().cpu().item()),
        "-phi'(0)": float(-php.detach().cpu().item()),
    }


def pinn_profiles(model, n=500):
    eta_np = np.linspace(0.0, ETA_MAX, n)
    eta = torch.tensor(eta_np, dtype=DTYPE, device=device).reshape(-1, 1)
    with torch.no_grad():
        out = model(eta).cpu().numpy()
    return eta_np, out[:, 0], out[:, 1], out[:, 2]


## 6. Train PINN

In [ ]:
def rel_error_pct(pred, ref):
    if abs(ref) < 1e-10:
        return np.nan
    return abs(pred-ref)/abs(ref)*100.0

os.makedirs("results", exist_ok=True)
table6_rows = []
profile_error_rows = []
models = {}
histories = {}

for case_name, changes in CASES.items():
    p = BASE.copy()
    p.update(changes)
    ref = reference_solutions[case_name][1]
    print(f"\\n===== {case_name} =====")
    model, history = train_pinn(p, adam_epochs=500, lbfgs_iterations=50, verbose=True)
    models[case_name], histories[case_name] = model, history

    q_ref = reference_wall_quantities(ref)
    q_pinn = pinn_wall_quantities(model)
    for quantity in ["f''(0)", "-theta'(0)", "-phi'(0)"]:
        table6_rows.append({
            "Parameter case": case_name,
            "Quantity": quantity,
            "PINN prediction": q_pinn[quantity],
            "Independent BVP reference": q_ref[quantity],
            "Relative error (%)": rel_error_pct(q_pinn[quantity], q_ref[quantity]),
            "BVP success": bool(ref.success),
            "BVP nodes": int(ref.x.size),
            "BVP max RMS residual": float(np.max(ref.rms_residuals)),
        })

    eta = np.linspace(0.0, ETA_MAX, 500)
    pinn_eta, f_p, th_p, ph_p = pinn_profiles(model, n=500)
    Yref = ref.sol(eta)
    for label, pred, truth in [
        ("f(eta)", f_p, Yref[0]),
        ("theta(eta)", th_p, Yref[3]),
        ("phi(eta)", ph_p, Yref[5]),
    ]:
        profile_error_rows.append({
            "Parameter case": case_name,
            "Profile": label,
            "MAE": float(np.mean(np.abs(pred-truth))),
            "RMSE": float(np.sqrt(np.mean((pred-truth)**2))),
            "Maximum absolute error": float(np.max(np.abs(pred-truth))),
        })
    history.to_csv(f"results/training_history_{case_name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('=', '')}.csv", index=False)

table6 = pd.DataFrame(table6_rows)
profile_errors = pd.DataFrame(profile_error_rows)
print("Table 6 — independent wall-quantity validation")
display(table6.round(7))
print("Profile errors")
display(profile_errors.round(7))
table6.to_csv("results/Table6_independent_validation.csv", index=False)
profile_errors.to_csv("results/Profile_error_metrics.csv", index=False)
print("Saved results/Table6_independent_validation.csv and results/Profile_error_metrics.csv")


## 7. Profile comparison plots

In [ ]:
for case_name, changes in CASES.items():
    p = BASE.copy()
    p.update(changes)
    ref = reference_solutions[case_name][1]
    model = models[case_name]
    eta = np.linspace(0.0, ETA_MAX, 500)
    Yref = ref.sol(eta)
    _, f_p, th_p, ph_p = pinn_profiles(model, n=500)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, ref_y, pinn_y, label in [
        (axes[0], Yref[0], f_p, r"$f(\eta)$"),
        (axes[1], Yref[3], th_p, r"$\theta(\eta)$"),
        (axes[2], Yref[5], ph_p, r"$\phi(\eta)$"),
    ]:
        ax.plot(eta, ref_y, label="Independent BVP", linewidth=2)
        ax.plot(eta[::10], pinn_y[::10], "o", label="PINN", markersize=3)
        ax.set_xlabel(r"$\eta$")
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3)
    axes[0].legend()
    fig.suptitle(case_name)
    fig.tight_layout()
    safe = case_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    fig.savefig(f"results/Table6_profiles_{safe}.png", dpi=300, bbox_inches="tight")
    plt.show()


## 8. BVP mesh/tolerance refinement check

In [8]:
p = BASE.copy()
ref_standard = reference_solutions["Baseline"][1]
ref_tight = solve_reference(p, tol=1e-9, n_initial=600)
q_std = reference_wall_quantities(ref_standard)
q_tight = reference_wall_quantities(ref_tight)
refine_rows = []
for qname in ["f''(0)", "-theta'(0)", "-phi'(0)"]:
    refine_rows.append({
        "Quantity": qname,
        "Standard BVP": q_std[qname],
        "Refined BVP": q_tight[qname],
        "Absolute difference": abs(q_std[qname]-q_tight[qname]),
        "Tight BVP max RMS residual": float(np.max(ref_tight.rms_residuals)),
    })
refinement_check = pd.DataFrame(refine_rows)
display(refinement_check.round(9))
refinement_check.to_csv("results/BVP_refinement_check.csv", index=False)


,Quantity,Standard BVP,Refined BVP,Absolute difference,Tight BVP max RMS residual
0,f''(0),-0.918320,-0.918320,0.0,1.000000e-09
1,-theta'(0),0.650014,0.650014,0.0,1.000000e-09
2,-phi'(0),0.473932,0.473932,0.0,1.000000e-09


## 9. Full-training collocation-point sensitivity study

In [ ]:
# Full training sensitivity experiment. The full schedule is intentional; do not shorten
# the run and report it as full-training convergence evidence.
from pathlib import Path
from IPython.display import display
import json

SENSITIVITY_POINT_COUNTS = [100, 200, 400, 1000]
SENSITIVITY_ADAM_EPOCHS = 3000
SENSITIVITY_LBFGS_ITERATIONS = 200
SENSITIVITY_DIR = Path("results/collocation_sensitivity")
SENSITIVITY_DIR.mkdir(parents=True, exist_ok=True)

# Recompute the baseline reference independently with the same parameter configuration.
sens_ref = solve_reference(BASE.copy(), tol=1e-8, n_initial=300)
sens_q_ref = reference_wall_quantities(sens_ref)
eta_eval = np.linspace(0.0, ETA_MAX, 501)
y_ref_eval = sens_ref.sol(eta_eval)

sens_rows = []
sens_profile_rows = []
sens_profiles = {}
sens_models = {}
sens_histories = {}

for n_points in SENSITIVITY_POINT_COUNTS:
    print(f"\n===== Full-training sensitivity: N={n_points} =====")
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    start_time = time.time()
    model, history = train_pinn(
        BASE.copy(),
        adam_epochs=SENSITIVITY_ADAM_EPOCHS,
        lbfgs_iterations=SENSITIVITY_LBFGS_ITERATIONS,
        verbose=True,
        n_points=n_points,
    )
    elapsed = time.time() - start_time
    sens_models[n_points] = model
    sens_histories[n_points] = history

    # The explicit n_points argument is also used for the reported final loss.
    final_total, final_ode, final_bc = pinn_loss(model, BASE, n_points=n_points)
    q_pinn = pinn_wall_quantities(model)
    row = {
        "Collocation points": n_points,
        "Adam epochs": SENSITIVITY_ADAM_EPOCHS,
        "L-BFGS max iterations": SENSITIVITY_LBFGS_ITERATIONS,
        "Elapsed time (s)": elapsed,
        "Final total loss": float(final_total.detach().cpu()),
        "Final ODE residual loss": float(final_ode.cpu()),
        "Final boundary loss": float(final_bc.cpu()),
    }
    for qname in ["f''(0)", "-theta'(0)", "-phi'(0)"]:
        pred, refval = q_pinn[qname], sens_q_ref[qname]
        row[f"PINN {qname}"] = pred
        row[f"BVP {qname}"] = refval
        row[f"Absolute error {qname}"] = abs(pred-refval)
        row[f"Relative error (%) {qname}"] = abs(pred-refval) / max(abs(refval), 1e-12) * 100.0

    eta, f_pred, th_pred, ph_pred = pinn_profiles(model, n=len(eta_eval))
    sens_profiles[n_points] = (eta, f_pred, th_pred, ph_pred)
    for label, pred, truth in [
        ("f(eta)", f_pred, y_ref_eval[0]),
        ("theta(eta)", th_pred, y_ref_eval[3]),
        ("phi(eta)", ph_pred, y_ref_eval[5]),
    ]:
        sens_profile_rows.append({
            "Collocation points": n_points,
            "Profile": label,
            "MAE vs BVP": float(np.mean(np.abs(pred-truth))),
            "RMSE vs BVP": float(np.sqrt(np.mean((pred-truth)**2))),
            "Maximum absolute error vs BVP": float(np.max(np.abs(pred-truth))),
        })

    history.to_csv(SENSITIVITY_DIR / f"training_history_N{n_points}.csv", index=False)
    torch.save(model.state_dict(), SENSITIVITY_DIR / f"pinn_N{n_points}.pt")
    sens_rows.append(row)
    pd.DataFrame(sens_rows).to_csv(SENSITIVITY_DIR / "Table_S1_wall_sensitivity.csv", index=False)
    pd.DataFrame(sens_profile_rows).to_csv(SENSITIVITY_DIR / "Table_S2_profile_errors.csv", index=False)
    print(json.dumps(row, indent=2))

# Percentage changes between successive point densities. This is a numerical
# comparison, not a substitute for checking the profile and BVP-error tables.
sens_table = pd.DataFrame(sens_rows)
for qname in ["f''(0)", "-theta'(0)", "-phi'(0)"]:
    sens_table[f"Change in {qname} vs previous density (%)"] = (
        sens_table[f"PINN {qname}"].diff().abs()
        / sens_table[f"PINN {qname}"].shift(1).abs().clip(lower=1e-12) * 100.0
    )
sens_table.to_csv(SENSITIVITY_DIR / "Table_S1_wall_sensitivity.csv", index=False)
sens_table.to_excel(SENSITIVITY_DIR / "Table_S1_wall_sensitivity.xlsx", index=False)
profile_table = pd.DataFrame(sens_profile_rows)
profile_table.to_csv(SENSITIVITY_DIR / "Table_S2_profile_errors.csv", index=False)
profile_table.to_excel(SENSITIVITY_DIR / "Table_S2_profile_errors.xlsx", index=False)

# Plot the three profiles for each density against the same independent BVP solution.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, ref_idx, profile_idx, label in [
    (axes[0], 0, 1, r"$f(\eta)$"),
    (axes[1], 3, 2, r"$\theta(\eta)$"),
    (axes[2], 5, 3, r"$\phi(\eta)$"),
]:
    ax.plot(eta_eval, y_ref_eval[ref_idx], "k-", linewidth=2.5, label="Independent BVP")
    for n_points, vals in sens_profiles.items():
        ax.plot(vals[0], vals[profile_idx], linewidth=1.2, label=f"PINN N={n_points}")
    ax.set_xlabel(r"$\eta$")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.25)
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(SENSITIVITY_DIR / "Figure_S1_collocation_profile_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nWall-quantity sensitivity table:")
display(sens_table.round(8))
print("\nProfile-error table against the independent BVP:")
display(profile_table.round(8))
print(f"Saved sensitivity tables, model checkpoints, training histories, and profile figure to {SENSITIVITY_DIR.resolve()}")


# 10. Sankari et al. (2023) benchmark reproduction — source-equation implementation

In [ ]:
# Published reference values transcribed from Sankari et al. (2023), Tables 1–3.
SANKARI_PUBLISHED = {
    "Skin friction": 0.936575,   # -f''(0), Table 1
    "Nusselt number": 0.783805,  # -theta'(0), Table 2
    "Sherwood number": 0.684867, # -phi'(0), Table 3
}

# Parameter settings explicitly listed for each table row.
# Remaining coupled parameters are held at the manuscript baseline and documented.
SANKARI_BASE = {
    "beta": 0.5, "M": 0.4, "Kp": 0.3,
    "Pr": 1.1, "Rd": 0.3, "Ec": 0.1,
    "Nb": 0.1, "Nt": 0.1, "St": 0.1, "Q": 0.1,
    "Sc": 0.2, "c": 0.1, "Cr": 0.2, "S": 0.2,
}
SANKARI_CASES = {
    "Skin friction": {**SANKARI_BASE},
    "Nusselt number": {**SANKARI_BASE},
    "Sherwood number": {**SANKARI_BASE, "Nb": 0.3, "Nt": 0.2},
}


## 11.1 Governing equations used for the Sankari benchmars

In [ ]:
def sankari_ode(eta, Y, p):
    f, fp, fpp, th, thp, ph, php = Y
    fppp = (2.0*fp**2 + (p["M"] + p["Kp"])*fp - f*fpp) / (1.0 + 1.0/p["beta"])
    thpp = -(
        p["Pr"] * (
            f*thp - fp*th - p["St"]*fp + p["Q"]*th
            + p["Nb"]*thp*php + p["Nt"]*thp**2
        )
        + p["Pr"]*p["Ec"] * (
            (1.0 + 1.0/p["beta"])*fpp**2 + p["M"]*fp**2
        )
    ) / (1.0 + 4.0*p["Rd"]/3.0)
    phpp = -p["Sc"] * (
        f*php - fp*ph - p["c"]*fp - p["Cr"]*ph**2
    ) - (p["Nt"]/p["Nb"])*thpp
    return np.vstack((fp, fpp, fppp, thp, thpp, php, phpp))


def sankari_bc(ya, yb, p):
    return np.array([
        ya[0] - p["S"], ya[1] - 1.0,
        ya[3] - (1.0-p["St"]), ya[5] - (1.0-p["Sc"]),
        yb[1], yb[3], yb[5]
    ])


def solve_sankari_bvp(p, eta_max=12.0, tol=1e-8, n_initial=500):
    x = np.linspace(0.0, eta_max, n_initial)
    y = np.zeros((7, x.size))
    y[0] = p["S"] + 1.0 - np.exp(-x)
    y[1] = np.exp(-x)
    y[2] = -np.exp(-x)
    y[3] = (1.0-p["St"])*np.exp(-x)
    y[4] = -(1.0-p["St"])*np.exp(-x)
    y[5] = (1.0-p["Sc"])*np.exp(-x)
    y[6] = -(1.0-p["Sc"])*np.exp(-x)
    sol = solve_bvp(
        lambda x, y: sankari_ode(x, y, p),
        lambda ya, yb: sankari_bc(ya, yb, p),
        x, y, tol=tol, max_nodes=50000
    )
    if not sol.success:
        raise RuntimeError(f"Sankari BVP failed: {sol.message}")
    return sol


def sankari_wall_quantities(sol):
    return {
        "Skin friction": float(-sol.y[2, 0]),
        "Nusselt number": float(-sol.y[4, 0]),
        "Sherwood number": float(-sol.y[6, 0]),
    }


def percent_error(value, reference):
    return abs(value-reference)/abs(reference)*100.0 if abs(reference) > 1e-12 else np.nan


sankari_bvp_solutions = {}
sankari_bvp_rows = []
for case_name, p in SANKARI_CASES.items():
    sol = solve_sankari_bvp(p)
    sankari_bvp_solutions[case_name] = sol
    q = sankari_wall_quantities(sol)
    quantity = case_name
    sankari_bvp_rows.append({
        "Quantity": quantity,
        "Published Sankari et al.": SANKARI_PUBLISHED[quantity],
        "Recomputed BVP": q[quantity],
        "Relative difference vs published (%)": percent_error(q[quantity], SANKARI_PUBLISHED[quantity]),
        "BVP success": bool(sol.success),
        "BVP nodes": int(sol.x.size),
        "Maximum RMS residual": float(np.max(sol.rms_residuals)),
        "Parameter assumptions": "Table-listed values + remaining manuscript-baseline values"
    })

sankari_bvp_table = pd.DataFrame(sankari_bvp_rows)
display(sankari_bvp_table.round(7))
sankari_bvp_table.to_csv("results/Sankari_BVP_benchmark.csv", index=False)


## 11.2 PINN training against the source equations


In [ ]:
class SankariPINN(nn.Module):
    def __init__(self, hidden_layers=6, neurons=64):
        super().__init__()
        layers = [nn.Linear(1, neurons), nn.Tanh()]
        for _ in range(hidden_layers-1):
            layers.extend([nn.Linear(neurons, neurons), nn.Tanh()])
        layers.append(nn.Linear(neurons, 3))
        self.net = nn.Sequential(*layers)

    def forward(self, eta):
        z = 2.0*eta/12.0 - 1.0
        return self.net(z)


def sankari_pinn_residuals(model, p, eta):
    eta = eta.clone().detach().requires_grad_(True)
    out = model(eta)
    f, th, ph = out[:, 0:1], out[:, 1:2], out[:, 2:3]
    fp = d_dx(f, eta); fpp = d_dx(fp, eta); fppp = d_dx(fpp, eta)
    thp = d_dx(th, eta); thpp = d_dx(thp, eta)
    php = d_dx(ph, eta); phpp = d_dx(php, eta)
    r_f = (1+1/p["beta"])*fppp + f*fpp - 2*fp**2 - (p["M"]+p["Kp"])*fp
    r_th = ((1+4*p["Rd"]/3)*thpp
        + p["Pr"]*(f*thp-fp*th-p["St"]*fp+p["Q"]*th
                   + p["Nb"]*thp*php+p["Nt"]*thp**2)
        + p["Pr"]*p["Ec"]*((1+1/p["beta"])*fpp**2+p["M"]*fp**2))
    r_ph = phpp + p["Sc"]*(f*php-fp*ph-p["c"]*fp-p["Cr"]*ph**2) + (p["Nt"]/p["Nb"])*thpp
    return r_f, r_th, r_ph


def sankari_pinn_loss(model, p, n_points=200):
    eta = torch.linspace(0.0, 12.0, n_points, device=device, dtype=DTYPE).reshape(-1,1)
    residuals = sankari_pinn_residuals(model, p, eta)
    loss_ode = sum(torch.mean(r**2) for r in residuals)
    e0 = torch.zeros((1,1), device=device, dtype=DTYPE, requires_grad=True)
    eL = torch.full((1,1), 12.0, device=device, dtype=DTYPE, requires_grad=True)
    o0, oL = model(e0), model(eL)
    fp0 = d_dx(o0[:,0:1], e0); fpL = d_dx(oL[:,0:1], eL)
    bc = [
        o0[:,0:1]-p["S"], fp0-1.0,
        o0[:,1:2]-(1-p["St"]), o0[:,2:3]-(1-p["Sc"]),
        fpL, oL[:,1:2], oL[:,2:3]
    ]
    loss_bc = sum(torch.mean(b**2) for b in bc)
    return loss_ode + 10.0*loss_bc, loss_ode.detach(), loss_bc.detach()


def train_sankari_pinn(p, adam_epochs=3000, lbfgs_iterations=200, verbose=True):
    model = SankariPINN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    logs = []
    for epoch in range(adam_epochs):
        opt.zero_grad()
        loss, lode, lbc = sankari_pinn_loss(model, p)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
        opt.step()
        if (epoch+1) % 250 == 0 or epoch == 0:
            logs.append([epoch+1, float(loss.detach().cpu()), float(lode.cpu()), float(lbc.cpu())])
            if verbose:
                print(f"Adam {epoch+1}/{adam_epochs}: total={loss.item():.3e}; ODE={lode.item():.3e}; BC={lbc.item():.3e}")
    opt2 = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iterations,
                             history_size=50, line_search_fn="strong_wolfe",
                             tolerance_grad=1e-9, tolerance_change=1e-12)
    def closure():
        opt2.zero_grad()
        loss, _, _ = sankari_pinn_loss(model, p)
        loss.backward()
        return loss
    opt2.step(closure)
    final, lode, lbc = sankari_pinn_loss(model, p)
    print(f"Final loss={final.item():.3e}; ODE={lode.item():.3e}; BC={lbc.item():.3e}")
    return model, pd.DataFrame(logs, columns=["epoch","total_loss","ODE_loss","BC_loss"])


def sankari_pinn_walls(model):
    e0 = torch.zeros((1,1), device=device, dtype=DTYPE, requires_grad=True)
    out = model(e0)
    f, th, ph = out[:,0:1], out[:,1:2], out[:,2:3]
    fp = d_dx(f,e0); fpp = d_dx(fp,e0)
    thp = d_dx(th,e0); php = d_dx(ph,e0)
    return {"Skin friction": float(-fpp.detach().cpu().item()),
            "Nusselt number": float(-thp.detach().cpu().item()),
            "Sherwood number": float(-php.detach().cpu().item())}


In [ ]:
# Full benchmark training. Run this cell to generate PINN results.
# For a quick smoke test only, reduce adam_epochs, but do not report that run as final validation.
sankari_models, sankari_histories = {}, {}
sankari_final_rows = []

for case_name, p in SANKARI_CASES.items():
    print("\\n===== Sankari benchmark:", case_name, "=====")
    model, hist = train_sankari_pinn(p, adam_epochs=3000, lbfgs_iterations=200, verbose=True)
    sankari_models[case_name], sankari_histories[case_name] = model, hist
    q_pinn = sankari_pinn_walls(model)
    q_bvp = sankari_wall_quantities(sankari_bvp_solutions[case_name])
    for qname in ["Skin friction", "Nusselt number", "Sherwood number"]:
        if qname == case_name:
            sankari_final_rows.append({
                "Quantity": qname,
                "Published Sankari et al.": SANKARI_PUBLISHED[qname],
                "BVP reference": q_bvp[qname],
                "PINN result": q_pinn[qname],
                "PINN vs BVP error (%)": percent_error(q_pinn[qname], q_bvp[qname]),
                "BVP vs published difference (%)": percent_error(q_bvp[qname], SANKARI_PUBLISHED[qname]),
                "PINN vs published difference (%)": percent_error(q_pinn[qname], SANKARI_PUBLISHED[qname]),
                "Final ODE loss": float(sankari_pinn_loss(model, p)[1].cpu()),
                "Final BC loss": float(sankari_pinn_loss(model, p)[2].cpu())
            })
    hist.to_csv(f"results/Sankari_training_{case_name.lower().replace(' ', '_')}.csv", index=False)

sankari_final_table = pd.DataFrame(sankari_final_rows)
display(sankari_final_table.round(7))
sankari_final_table.to_csv("results/Sankari_PINN_BVP_published_comparison.csv", index=False)
